In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import os
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import shutil

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zaryabahmadkhan/2d-slicing-of-imagetbad-dataset")

print("Path to dataset files:", path)

100%|██████████| 2.42G/2.42G [02:00<00:00, 21.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/zaryabahmadkhan/2d-slicing-of-imagetbad-dataset/versions/1


In [3]:
path

'/root/.cache/kagglehub/datasets/zaryabahmadkhan/2d-slicing-of-imagetbad-dataset/versions/1'

In [4]:
!pip install hydra-core omegaconf
import hydra
from omegaconf import DictConfig, OmegaConf
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.9 MB/s eta 0:00:00


In [5]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.9 MB/s eta 0:00:00


In [6]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
%env CLEARML_API_ACCESS_KEY=XGMS61QRIXH7G4URU0X7VGTDAIQUVE
%env CLEARML_API_SECRET_KEY=3mpOB_Br37BLBntBoUvtxmfbBfmkbkdfRUYGje8GfL1gwe_ufJd1w4UzuQWOPaAtTWE

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=XGMS61QRIXH7G4URU0X7VGTDAIQUVE
env: CLEARML_API_SECRET_KEY=3mpOB_Br37BLBntBoUvtxmfbBfmkbkdfRUYGje8GfL1gwe_ufJd1w4UzuQWOPaAtTWE


In [7]:
from clearml import Task, Logger

In [31]:
from google.colab import drive
drive.mount('/content/drive')
config_dir = '/content/drive/MyDrive/segmentation_project/config'
import shutil
def config():
    config_name = "config"
    with hydra.initialize_config_dir(config_dir=config_dir, version_base="1.2"):
        cfg = hydra.compose(config_name=config_name)
    OmegaConf.set_struct(cfg, False)
    return cfg

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
def split():
    cfg = config()
    im_path = os.path.join(path, '2D dataset', 'images')
    lb_path = os.path.join(path, '2D dataset', 'labels')
    output = cfg.data.output
    pairs = []
    for file in os.listdir(im_path):
        if file.endswith('_image.png'):
            lb_file = file.replace('_image.png', '_label.png')
            if os.path.exists(os.path.join(lb_path, lb_file)):
                pairs.append((file, lb_file))
    train_pairs, test_pairs = train_test_split(
        pairs,
        test_size=cfg.data.test_size,
        random_state=cfg.data.random_state
    )
    for cat, pair in [('train', train_pairs), ('test', test_pairs)]:
        img_dir = os.path.join(output,cat, 'images')
        lbl_dir = os.path.join(output,cat, 'labels')
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)
        for img_file, lbl_file in pair:
            shutil.copy2(
                os.path.join(im_path, img_file),
                os.path.join(img_dir, img_file)
            )
            shutil.copy2(
                os.path.join(lb_path, lbl_file),
                os.path.join(lbl_dir, lbl_file)
            )
    return {
        'train_images': os.path.join(output, 'train', 'images'),
        'train_labels': os.path.join(output, 'train', 'labels'),
        'test_images': os.path.join(output, 'test', 'images'),
        'test_labels': os.path.join(output, 'test', 'labels')
    }

In [32]:
class Dataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.png')])
        self.label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith('.png')])
    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, self.label_files[idx])
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
        if self.transform:
            transformed = self.transform(image=image, mask=label)
            image = transformed['image']
            label = transformed['mask']
        label = (label > 0).long()

        return image, label

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.enc1 = self._contract(in_channel,64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self._contract(64,128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self._contract(128,256)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256,512,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.Conv2d(512,512,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.ConvTranspose2d(512,256,2,stride=2)
        )
        self.dec3 = self._exp(512,256,128)
        self.dec2 = self._exp(256,128,64)
        self.fc = self._final(128,64,out_channel)

        self._init_weights()

    def _contract(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels,out_channels,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_channels),
            nn.Conv2d(out_channels, out_channels,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_channels)
        )
    def _exp(self, in_channels, hidden_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels,hidden_channels,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            nn.Conv2d(hidden_channels,hidden_channels,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            nn.ConvTranspose2d(hidden_channels,out_channels,2,stride=2)
        )

    def _final(self, in_channels, hidden_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels,hidden_channels,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            nn.Conv2d(hidden_channels,hidden_channels,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_channels),
            nn.Conv2d(hidden_channels,out_channels,3,padding=1)
        )

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight,1)
                nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.ConvTranspose2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias,0)

    def _c_c(self, up, down):
        _, _, hu, wu = up.shape
        _, _, hb, wb = down.shape
        ch = (hb - hu) // 2
        cw = (wb - wu) // 2
        cropped = down[:, :, ch:ch+hu, cw:cw+wu]
        return torch.cat([up, cropped], dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        e3 = self.enc3(p2)
        p3 = self.pool3(e3)
        b = self.bottleneck(p3)
        d3 = self._c_c(b, e3)
        c2 = self.dec3(d3)
        d2 = self._c_c(c2, e2)
        c1 = self.dec2(d2)
        d1 = self._c_c(c1, e1)
        out = self.fc(d1)
        return out

In [12]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

def transformers():
    cfg = config()
    train_transform = A.Compose([
        A.Resize(height=cfg.transforms.image_size, width=cfg.transforms.image_size),

        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={'x': (-0.05, 0.05), 'y': (-0.05, 0.05)},
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            p=0.5
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.1,
            contrast_limit=0.1,
            p=0.3
        ),
        A.GaussNoise(p=0.2),

        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ])

    val_transform = A.Compose([
        A.Resize(height=cfg.transforms.image_size, width=cfg.transforms.image_size),
        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ])

    return train_transform, val_transform

In [13]:
!pip install torchmetrics
from torchmetrics.segmentation import DiceScore
from torchmetrics import JaccardIndex
from torchmetrics import Precision, Recall


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 29.5 MB/s eta 0:00:00


In [33]:
cfg = config()

def settings(cfg):
    task = Task.init(
        project_name='JustS',
        task_name=f'UNet_20epochs',
        auto_connect_frameworks={'pytorch': True, 'hydra': True}
    )
    cfg_dict = OmegaConf.to_container(cfg,resolve=True)
    task.connect_configuration(cfg_dict)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    paths = split()
    train_transform, val_transform = transformers()
    train_dataset = Dataset(
        images_dir=paths['train_images'],
        labels_dir=paths['train_labels'],
        transform=val_transform
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.training.batch_size,
        shuffle=True,
        num_workers=cfg.data.num_workers
    )
    test_dataset = Dataset(
        images_dir=paths['test_images'],
        labels_dir=paths['test_labels'],
        transform=val_transform
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.training.batch_size,
        shuffle=False,
        num_workers=cfg.data.num_workers
    )
    model = UNet(cfg.model.in_channel, cfg.model.out_channel).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.training.learning_rate,
        weight_decay=cfg.training.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min')
    metrics = {
        'dice': DiceScore(num_classes=2, average='macro', input_format='index').to(device),
        'iou': JaccardIndex(num_classes=2, task='multiclass', average='macro').to(device),
        'precision': Precision(num_classes=2, task='multiclass', average='macro').to(device),
        'recall': Recall(num_classes=2, task='multiclass', average='macro').to(device)
    }
    return {
        'device': device,
        'model': model,
        'test_loader': test_loader,
        'train_loader': train_loader,
        'criterion': criterion,
        'optimizer': optimizer,
        'scheduler': scheduler,
        'metrics': metrics,
        'task': task

    }

In [34]:
def train_epoch(model,train_loader,criterion,optimizer,device):
  model.train()
  epoch_loss = 0
  train_pbar = tqdm(train_loader,desc='Train')
  for idx, (images, labels) in enumerate(train_pbar):
    images, labels = images.to(device), labels.to(device)
    labels = labels.long()
    optimizer.zero_grad()
    output = model(images)
    loss = criterion(output, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    epoch_loss += loss.item()
    train_pbar.set_postfix({
        'loss': f'{epoch_loss/(idx+1):.4f}'
    })
  train_pbar.close()
  avg_loss = epoch_loss / len(train_loader)
  return avg_loss

In [35]:
def validate(model, test_loader, metrics, device):
    model.eval()
    for metric in metrics.values():
        metric.reset()
    val_pbar = tqdm(test_loader, desc='Validation')
    with torch.no_grad():
        for idx, (images, masks) in enumerate(val_pbar):
            images, masks = images.to(device), masks.to(device)
            masks = masks.long()
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            for metric in metrics.values():
                metric.update(preds, masks)
    val_pbar.close()
    new_metrics = {}
    for name, metric in metrics.items():
        new_metrics[name] = metric.compute().item()
    return new_metrics

In [39]:
def early_stopping(cur_metric, best_metric, patience_count, patience, min_d):
    better = False
    if cur_metric < best_metric - min_d:
      better = True
    else:
      if cur_metric > best_metric + min_d:
            better = True
    if better:
        patience_count = 0
    else:
        patience_count += 1
    stop = patience_count >= patience
    return stop, patience_count

In [40]:
from pathlib import Path
def train_model():
    setup = settings(cfg)
    device = setup['device']
    model = setup['model']
    train_loader = setup['train_loader']
    test_loader = setup['test_loader']
    criterion = setup['criterion']
    optimizer = setup['optimizer']
    scheduler = setup['scheduler']
    metrics = setup['metrics']
    task = setup['task']
    patience = cfg.early_stopping.patience
    min_delta = cfg.early_stopping.min_delta
    restore_best = cfg.early_stopping.restore_best_weights

    best_dice = 0
    best_epoch = 0
    patience_count = 0
    logger = task.get_logger()
    checkpoint_dir = Path('checkpoints')
    checkpoint_dir.mkdir(exist_ok=True)

    for epoch in range(cfg.training.epochs):
        avg_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        new_metrics = validate(model, test_loader, metrics, device)

        if new_metrics['dice'] > best_dice:
            best_dice = new_metrics['dice']
            best_epoch = epoch
        logger.report_scalar("Loss", "Train", avg_loss, epoch)
        logger.report_scalar("Metrics", "dice", new_metrics['dice'], epoch)
        logger.report_scalar("Metrics", "iou", new_metrics['iou'], epoch)
        logger.report_scalar("Metrics", "precision", new_metrics['precision'], epoch)
        logger.report_scalar("Metrics", "recall", new_metrics['recall'], epoch)
        scheduler.step(avg_loss)
        print(f'Epoch {epoch}:')
        print(f'  Loss: {avg_loss}')
        print(f'  Dice: {new_metrics["dice"]}')
        print(f'  Iou: {new_metrics["iou"]}')
        print(f'  Precision: {new_metrics["precision"]}')
        print(f'  Recall: {new_metrics["recall"]}')
        print(f'Best Dice {best_dice} Epoch {best_epoch}')
        stop, patience_count = early_stopping(
                new_metrics['dice'],
                best_metric=best_dice,
                patience_count=patience_count,
                patience=patience,
                min_d=min_delta
        )
        if stop:
            print(f"early stopping")
            break
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_loss,
            'new_metrics': new_metrics,
            'best_dice': best_dice,
            'best_epoch': best_epoch,
            'config': OmegaConf.to_container(cfg)
        }
        torch.save(checkpoint, checkpoint_dir / 'last_checkpoint.pth')
    print(f"Best epoch: {best_epoch} Dice: {best_dice:.4f}")
    return model

In [ ]:
if __name__ == '__main__':
  train_model()


Train:  14%|█▎        | 150/1093 [01:47<11:07,  1.41it/s, loss=0.4656]